In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

from google.colab import files

# Upload your original CSV
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Saving Hospital_Dataset_Final.csv to Hospital_Dataset_Final.csv
Dataset loaded successfully!
Rows: 5000
Columns: 42


In [ ]:
total_admissions = df["Case_No"].nunique()

print("Total Admissions:", total_admissions)

Total Admissions: 5000


In [ ]:
df["Total_Beds"] = pd.to_numeric(
    df["Total_Beds"],
    errors="coerce"
)

df["Occupied_Beds"] = pd.to_numeric(
    df["Occupied_Beds"],
    errors="coerce"
)

valid_beds = df[
    (df["Total_Beds"] > 0) &
    (df["Occupied_Beds"] >= 0) &
    (df["Occupied_Beds"] <= df["Total_Beds"])
].copy()

valid_beds["Occupancy_Calculated"] = (
    valid_beds["Occupied_Beds"] /
    valid_beds["Total_Beds"]
) * 100

occupancy_rate = valid_beds[
    "Occupancy_Calculated"
].mean()

print(
    f"Occupancy Rate: {occupancy_rate:.2f}%"
)

Occupancy Rate: 64.90%


In [ ]:
df["LOS_Admission"] = pd.to_numeric(
    df["LOS_Admission"],
    errors="coerce"
)

valid_los = df[
    df["LOS_Admission"] >= 0
]

average_los = valid_los[
    "LOS_Admission"
].mean()

print(
    f"Average Length of Stay: {average_los:.2f} days"
)

Average Length of Stay: 1.12 days


In [ ]:
known_readmission = df["Readmitted"].isin(["Yes", "No"])

readmission_rate = (
    (df.loc[known_readmission, "Readmitted"] == "Yes").sum()
    / known_readmission.sum()
) * 100

print(f"Readmission Rate: {readmission_rate:.2f}%")

Readmission Rate: 12.00%


In [ ]:
total_occupied = df["Occupied_Beds"].sum()
total_beds = df["Total_Beds"].sum()

bed_utilization_rate = (
    total_occupied / total_beds
) * 100

print(f"Bed Utilization Rate: {bed_utilization_rate:.2f}%")

Bed Utilization Rate: 64.42%


In [ ]:
#Create Department Metrics
df["Readmission_Flag"] = np.where(
    df["Readmitted"] == "Yes",
    1,
    np.where(
        df["Readmitted"] == "No",
        0,
        np.nan
    )
)

department = df.groupby("Specialty").agg(
    Admissions=("Case_No", "nunique"),
    Average_LOS=("LOS_Admission", "mean"),
    Occupancy_Rate=("Occupancy_Rate", "mean"),
    Readmission_Rate=("Readmission_Flag", "mean")
).reset_index()

department["Readmission_Rate"] = (
    department["Readmission_Rate"] * 100
)

department

,Specialty,Admissions,Average_LOS,Occupancy_Rate,Readmission_Rate
0,Cardiology,501,1.103792,64.153932,11.156187
1,Emergency,502,1.125498,68.220239,11.491935
2,Family Medicine,473,1.069767,67.026089,12.153518
3,Gastroenterology,497,1.084507,67.397223,13.905930
4,General Surgery,500,1.114000,61.772520,12.195122
5,Internal Medicine,483,1.217391,66.526211,12.526096
6,Nephrology,546,1.098901,65.537234,11.970534
7,Obstetrics & Gynecology,516,1.124031,63.283353,13.043478
8,Orthopaedics,505,1.176238,61.673842,10.821643
9,Pulmonology,477,1.123690,63.504486,10.759494


In [ ]:
#Normalize the three performence measures
def min_max_score(series):
    return (
        (series - series.min())
        /
        (series.max() - series.min())
    ) * 100


# Higher occupancy = better
department["Occupancy_Score"] = min_max_score(
    department["Occupancy_Rate"]
)

# Lower LOS = better
department["LOS_Score"] = (
    100 -
    min_max_score(department["Average_LOS"])
)

# Lower readmission = better
department["Readmission_Score"] = (
    100 -
    min_max_score(department["Readmission_Rate"])
)

In [ ]:
#Calculate Efficiency Score
department["Efficiency_Score"] = (
    0.40 * department["Occupancy_Score"]
    +
    0.30 * department["LOS_Score"]
    +
    0.30 * department["Readmission_Score"]
)

department["Efficiency_Score"] = (
    department["Efficiency_Score"]
    .round(2)
)

department = department.sort_values(
    "Efficiency_Score",
    ascending=False
)

department

,Specialty,Admissions,Average_LOS,Occupancy_Rate,Readmission_Rate,Occupancy_Score,LOS_Score,Readmission_Score,Efficiency_Score
1,Emergency,502,1.125498,68.220239,11.491935,100.000000,62.248267,76.721547,81.69
2,Family Medicine,473,1.069767,67.026089,12.153518,81.758666,100.000000,55.695139,79.41
6,Nephrology,546,1.098901,65.537234,11.970534,59.015556,80.264941,61.510735,66.14
0,Cardiology,501,1.103792,64.153932,11.156187,37.884815,76.951576,87.392312,64.46
3,Gastroenterology,497,1.084507,67.397223,13.905930,87.427960,90.015435,0.000000,61.98
9,Pulmonology,477,1.123690,63.504486,10.759494,27.964156,63.473191,100.000000,60.23
5,Internal Medicine,483,1.217391,66.526211,12.526096,74.122747,0.000000,43.853874,42.81
4,General Surgery,500,1.114000,61.772520,12.195122,1.507370,70.036986,54.372887,37.93
8,Orthopaedics,505,1.176238,61.673842,10.821643,0.000000,27.877390,98.024762,37.77
7,Obstetrics & Gynecology,516,1.124031,63.283353,13.043478,24.586212,63.242009,27.410441,37.03
